## Tools
### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
### 1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
### 2. A function or coroutine to execute.

In [21]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots "talk" because they are **excellent vocal mimics**. Here’s a more detailed explanation:\n\n### 1. **Natural Mimicry Skills**\n- **Wild parrots use mimicry** naturally. In the wild, parrots imitate the calls of their flock members. This helps:\n  - **Strengthen social bonds**\n  - **Identify each other**\n  - **Warn about threats**\n\n### 2. **Intelligent & Social Animals**\n- Parrots are **highly intelligent** and **very social** creatures. When kept as pets, they view their human owners as part of their “flock.”\n- Because humans talk, parrots **try to join in** and communicate with their new “flock” by mimicking human speech.\n\n### 3. **Not True Talking**\n- Parrots don’t **understand language** the way humans do, but they can **associate some words or phrases with actions, objects, or responses**.\n- They do not have vocal cords. Instead, **they control the muscles in their syrinx** (a vocal organ at the base of the trachea) to create a wide variety of so

In [22]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """ Get the weather at a location"""
    return f"It is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [23]:
response = model_with_tools.invoke("What's the weather like in Boston")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool name : {tool_call['name']}")
    print(f"Tool name : {tool_call['args']}")



content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 50, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_62f6f64af5', 'id': 'chatcmpl-DyoomHD9M3AvozUYg1M4Gww68YIpg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f3a2c-17b4-73f2-824e-6ecc94d24cc7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_8mHaHbMsFvLJAYgIvFNlyPQb', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 14, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool name : get_weather
Tool na

In [24]:
#step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3: Pass results back to model for final response

final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Boston is currently sunny. If you need more details such as the temperature or a forecast, let me know!


In [25]:
messages

[{'role': 'user', 'content': "What's the weather in Boston"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 49, 'total_tokens': 63, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_62f6f64af5', 'id': 'chatcmpl-DyoonubbSwRW4BjdAAHenDLoM7gTf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f3a2c-1b69-7fc2-89ca-7ef6c8b7b8d8-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_D9uRuRZBFt2sOIhRTHnPu5Bf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 14, 'total_tokens': 63, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

## Practice 1

In [26]:




import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1")

response = model.invoke("print first call to the model")

print(response.content)



If you want to print/log the first call made to a model (e.g., an OpenAI API call in Python), you typically need to set up a mechanism to intercept the call and print the relevant input/output. Here are some approaches depending on your context:

### **1. For OpenAI's API (with openai library, Python example):**

```python
import openai

first_call = True

def call_model(prompt):
    global first_call
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    if first_call:
        print("First call to model:")
        print("Prompt:", prompt)
        print("Response:", response['choices'][0]['message']['content'])
        first_call = False
    return response['choices'][0]['message']['content']

# Example usage
call_model("Hello, how are you?")
call_model("What is the weather today?")
```

---

### **2. With a class-based model wrapper:**

```python
class MyModel:
    def __init__(self):
        self.ha

In [27]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """ This give weather of the location"""
    return f"The weather is sunny in {location}"


model_with_tool = model.bind_tools([get_weather])

In [28]:
# executing tool

messages = [
    {
        "role":"user",
        "content": "What is the weather in India"
        }]

ai_message = model_with_tool.invoke(messages)
messages.append(ai_message)

print(messages)

print("We have done Tool Call but still not geting the response we have to execute the tool")





[{'role': 'user', 'content': 'What is the weather in India'}, AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 50, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_bc01cf4baf', 'id': 'chatcmpl-Dyoos3iHpPf22uIxJxqi7TDy6Rkxc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f3a2c-319d-7a31-9b11-42315a777e8e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'India'}, 'id': 'call_oZQIdvYAvjXJmXKFgZfV5sEX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 14, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_

In [29]:
for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tool.invoke(messages)
print(final_response.text)

The weather in India is currently sunny. If you need details for a specific city or region within India, please let me know!


## practice 2

In [30]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1")

response = model.invoke("Print the sentence Initial model invoke without the tool call")

print(response.content)

Initial model invoke without the tool call


In [31]:
from langchain.tools import tool

@tool
def get_wether(location:str)->str:
    """ Give loactions weather"""
    return f"It is sunny in {location}"

model_with_tool = model.bind_tools([get_weather])

print("Defined the tool and bind it with the model")

Defined the tool and bind it with the model


In [32]:
messages = [
    {
        "role":"user",
        "content": "What is the weather in India"
    }
]

ai_message = model_with_tool.invoke(messages)

messages.append(ai_message)


for tool_call in ai_message.tool_calls:
    print(tool_call)

    tool_result = get_weather.invoke(tool_call)
    print(tool_result)
    messages.append(tool_result)


final_response = model_with_tool.invoke(messages)
print(final_response)

{'name': 'get_weather', 'args': {'location': 'India'}, 'id': 'call_kgJlOr5j3mLeM76IcwWohFOm', 'type': 'tool_call'}
content='The weather is sunny in India' name='get_weather' tool_call_id='call_kgJlOr5j3mLeM76IcwWohFOm'
content='The weather in India is currently sunny. If you need information for a specific city or region in India, please let me know!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 78, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_bc01cf4baf', 'id': 'chatcmpl-DyoowfnQFG0GiPBy837Evn4BrxY1u', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f3a2c-3dfe-7ac3-85d2-ad3bbe0e74aa-0' tool_calls=[] invalid_tool_ca